# BraTS-GLI: download and preprocess (CPU)

Companion notebook for [MHKhan50/brats-glioma-segmentation](https://github.com/MHKhan50/brats-glioma-segmentation).
Downloads the BraTS 2023 adult glioma (BraTS-GLI) training data from Synapse with **your own access** and prepares it for training.
The data are not redistributed: keep this notebook and its output **private**.

Before running: Synapse → Account Settings → Personal Access Tokens → create a token with *view* and *download*; in Kaggle,
*Add-ons → Secrets* → add `SYNAPSE_AUTH_TOKEN` and attach it to this notebook.

Kaggle settings: **CPU**, **Internet on**, then *Save Version → Save & Run All* (about 1.5–2 hours).
Output: `/kaggle/working/prep/` (about 13 GB): one uint8 array per case (4 MRI channels cropped to the brain, resampled to 128³, z-scored; plus labels),
the original-resolution labels for evaluation in native space, and `cases.json` with the patient-level split (70/10/20).

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("brats-glioma-segmentation"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/brats-glioma-segmentation.git"], check=True)
os.chdir("brats-glioma-segmentation")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
def bs(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "brats_seg.cli", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["SYNAPSE_AUTH_TOKEN"] = UserSecretsClient().get_secret("SYNAPSE_AUTH_TOKEN")
RAW, PREP = "/tmp/brats_raw", "/kaggle/working/prep"
bs("download", "--dest", RAW)

In [ ]:
bs("preprocess", "--raw", RAW, "--out", "/tmp/smoke", "--limit", "6", "--size", "64")   # smoke test
import shutil; shutil.rmtree("/tmp/smoke", ignore_errors=True)
bs("preprocess", "--raw", RAW, "--out", PREP, "--workers", "4")
shutil.copy(RAW + "/download_manifest.json", PREP + "/download_manifest.json")
shutil.rmtree(RAW, ignore_errors=True)
print(subprocess.run(["du", "-sh", PREP], capture_output=True, text=True).stdout)